# Building a spam filter based on a data set of emails (with machine learning)

The code in this notebook uses a data set of emails to find a rule for classifying an email as "spam" or "ham" (not spam). The data set is a collection of over 33,000 emails sent by employees of Enron. The emails have all been labelled as either "spam" or "ham". For more information see: https://www.kaggle.com/datasets/marcelwiechmann/enron-spam-data
[](https://www.kaggle.com/datasets/marcelwiechmann/enron-spam-data)

## Importing libraries and data

In [ ]:
# import pandas
import pandas as pd

# import machine learning functions from sklearn
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# There are currently some unhelpful warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# import functions for plotting the decision tree
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.metrics import accuracy_score

# Function to draw the model
def plot_decision_tree(tree_model):
    fig, ax = plt.subplots(figsize=(10,6))
    plot_tree(tree_model,  
        filled=True, 
        impurity=False, 
        feature_names= vectorizer.get_feature_names_out(), 
        class_names=["No","Yes"], 
        proportion=True, 
        ax=ax)
    plt.show()

In [ ]:
# import data
email_data = pd.read_csv('/kaggle/input/enron-spam-data/enron_spam_data.csv')

# display to check
email_data.head(5)

The data set contains 33,716 records, each representing a single email. 

Features:
* **Subject**: 	The subject line of the e-mail
* **Message**:	The content of the e-mail. Can contain an empty string if the message had only a subject line and no body. In case of forwarded emails or replies, this also contains the original message with subject line, "from:", "to:", etc.
* **Spam/Ham**:	Has the values "spam" or "ham". Whether the message was categorized as a spam message or not.
* **Date**:	The date the e-mail arrived. Has a YYYY-MM-DD format.

The data types and size of the data set can be displayed with `.info()`

In [ ]:
# explore the dataset
email_data.info()

# Preparing data
## Removing punctuation

In this notebook the text in the subject feature will be explored. The code below removes any punctiation from the subject feature and converts all letters to lower case.

In [ ]:
# Use re to prep the text - remove punctuation or capitals
email_data['Subject'] = email_data['Subject'].str.replace(r'[^\w\s]',' ', regex=True).str.lower()

The binary classification machine learning alogrithm will use a binary feature encoded as 0 or 1. The code in the boxes below creates a new feature with 1 for emails that are spam and 0 for emails that are spam. The counts of the `Spam/Ham` and `Spam` features can be displayed as a check.

In [ ]:
# replace categories with numbers
email_data['Spam?'] = email_data['Spam/Ham'].replace({'spam':1,'ham':0}) 

# display to check
email_data.head(5)

In [ ]:
email_data['Spam/Ham'].value_counts()

In [ ]:
email_data['Spam?'].value_counts()

# Building a model with machine learning
The machine learning algorithm uses a score for each word based on its frequency in the specific string (*Term Frequency*). This is created using a *vectorizer* that converts the string into a vector of values for the frequency of each word.

The data is split into *training* and *testing*. The training data is used to generate a decision tree of depth 3. The model from the decision tree is applied to the testing data to generate a *confusion matrix* and measure the *accuracy* of the predictions.

In [ ]:
# define the input and output variables 
X = vectorizer.fit_transform(email_data['Subject'])
y = email_data['Spam?']
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=1)

# create the model with the training data and display
tree_model = DecisionTreeClassifier(max_depth=3)
tree_model.fit(X_train, y_train)
plot_decision_tree(tree_model)

# create some predictions with the testing data and display the metrics
y_pred = tree_model.predict(X_test)
print(pd.crosstab(y_test, y_pred, rownames=['Actual'], colnames=['Predicted'], margins=True))
print("Accuracy: ",round(100*accuracy_score(y_test, y_pred),1),"%")

## Try some sentences
The code boxes below predict whether a given sample subject line would be predicted as spam or ham.

In [ ]:
# sample sentence
sample_subject = ["pay the transfers now"]

# create and display prediction
prediction = tree_model.predict(vectorizer.transform(sample_subject))
print(f"'{sample_subject}' → {prediction}")

In [ ]:
# sample sentence
sample_subject = ["pay the transfer now"]

# create and display prediction
prediction = tree_model.predict(vectorizer.transform(sample_subject))
print(f"'{sample_subject}' → {prediction}")